In [7]:
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain.agents import create_agent
from langgraph.checkpoint.memory import MemorySaver
import os
from dotenv import load_dotenv

load_dotenv()  # reads GOOGLE_API_KEY and ENDPOINT_URL from .env

GOOGLE_API_KEY = os.getenv("GOOGLE_API_KEY")
ENDPOINT_URL   = os.getenv("ENDPOINT_URL")

# Create Gemini LLM instance
llm = ChatGoogleGenerativeAI(
    model="gemini-3.5-flash",
    api_key=GOOGLE_API_KEY,
    base_url=ENDPOINT_URL,
    additional_headers={
        "Authorization": f"Bearer {GOOGLE_API_KEY}",
        "api-key": GOOGLE_API_KEY,
    },
    temperature=1,
    max_tokens=1024,
)
# Create memory checkpoint
memory = MemorySaver()
print(" Gemini LLM initialized.")
print(" MemorySaver checkpoint created.")


 Gemini LLM initialized.
 MemorySaver checkpoint created.


In [8]:

def draft_email(recipient_list: str, subject: str, body: str) -> str:

    """Draft an email to a list of friends requesting their availability."""

    return f"""Email drafted:

To: {recipient_list}

Subject: {subject}

Body: {body}

[Ready to send]"""

def record_response(friend_name: str, availability: str) -> str:

    """Record a friend's availability response."""

    return f"Recorded {friend_name}'s response: {availability}"

def book_restaurant(name: str, date: str, time: str, party_size: int) -> str:

    """Book a restaurant table."""

    return f"Restaurant '{name}' booked for {party_size} people on {date} at {time}"

print("Tools defined: draft_email, record_response, book_restaurant")


Tools defined: draft_email, record_response, book_restaurant


In [9]:

system_prompt = """Role: Professional butler, organizing social events and dinners.

Your Process:

1. Understand the user's event details (who, when, where preferences).
2. Draft an email invitation to friends requesting their availability.
3. Once you receive availability confirmation, consolidate responses.
4. Draft a restaurant booking call script with confirmed details.
5. Book the restaurant with final details (date, time, party size).
6. Draft a final confirmation email to all friends with venue, date, and time.

Key Behaviors:
- Break complex tasks into distinct steps.
- Use tools to draft communications and confirmations.
- Always confirm details before booking.
- Maintain a professional and organized tone.

Constraints:
- Do NOT book without confirmation.
- Do NOT send emails without drafting first.
- Always show your work and explain each step."""

# Create ReAct agent with memory

agent = create_agent(
    model=llm,
    tools=[draft_email, record_response, book_restaurant],
    checkpointer=memory,
    system_prompt=system_prompt,
)

print(" Scheduling agent created with MemorySaver.")

 Scheduling agent created with MemorySaver.


In [10]:
thread_id = "butler_session_001"
config = {"configurable": {"thread_id": thread_id}}
print(f" Thread ID: {thread_id}")
print(" Configuration ready.")

 Thread ID: butler_session_001
 Configuration ready.


In [11]:
from langchain_core.messages import AIMessage


def format_conversation(result) -> str:
    """Extract the agent's latest reply from the result as plain text."""
    messages = result.get("messages", [])
    for msg in reversed(messages):
        if isinstance(msg, AIMessage):
            # .text joins Gemini's content parts into one string
            # (content can be a plain string or a list of {"text": ...} parts).
            return msg.text
    return "No response generated."

print(" Message formatter ready.")


 Message formatter ready.


In [12]:
initial_request = {
    "messages": [
        {
            "role": "user",
            "content": "I want to organize a dinner with my 5 college friends. Can you help me set this up?"
        }
    ]

}

print("=" * 60)
print("TURN 1: Initial Request")
print("=" * 60)

print("User: I want to organize a dinner with my 5 college friends. Can you help me set this up?")

print()

result = agent.invoke(initial_request, config)
response = format_conversation(result)
print("Agent:")
print(response)
print()

print(f"Checkpoint: {len(result['messages'])} messages saved in thread '{thread_id}'")


TURN 1: Initial Request
User: I want to organize a dinner with my 5 college friends. Can you help me set this up?

Agent:
Good day! It would be my absolute pleasure to assist you in organizing a delightful dinner with your college friends. 

To ensure everything is arranged seamlessly, I will follow a structured step-by-step process:

1. **Understand Event Details** (Current Step)
2. **Draft Email Invitation** requesting availability from your friends
3. **Consolidate Responses** once availability is received
4. **Draft Restaurant Booking Script** with confirmed details
5. **Book Restaurant Table**
6. **Draft Final Confirmation Email** to all guests

***

### Step 1: Gathering Event Details

To begin, could you kindly provide me with the following information?

* **Friends' Contact Details:** Names and email addresses for your 5 college friends.
* **Date Preferences:** Proposed dates or a specific weekend/timeframe you have in mind.
* **Venue/Cuisine Preferences:** Any specific restaur

In [13]:

second_message = {
    "messages": [
        {
            "role": "user",
            "content": "Great! They all confirmed for next Friday evening around 7 PM. My friends are: Alice, Bob, Carol, Dave, and Eve. Please book a table for 6 at an Italian restaurant."
        }
    ]

}

print("=" * 60)
print("TURN 2: User Confirms Availability")
print("=" * 60)

print("User: Great! They all confirmed for next Friday evening around 7 PM. My friends are: Alice, Bob, Carol, Dave, and Eve. Please book a table for 6 at an Italian restaurant.")

print()
result = agent.invoke(second_message, config)
response = format_conversation(result)
print("Agent:")
print(response)
print()
print(f"Checkpoint: {len(result['messages'])} messages saved")
print("Agent has full context from Turn 1 (multi-turn memory working).")


TURN 2: User Confirms Availability
User: Great! They all confirmed for next Friday evening around 7 PM. My friends are: Alice, Bob, Carol, Dave, and Eve. Please book a table for 6 at an Italian restaurant.

Agent:
Thank you for providing those details. I have recorded the availability for all 5 of your friends (Alice, Bob, Carol, Dave, and Eve). 

Here is the progress update:

### Step 3: Response Consolidation
* **Total Party Size:** 6 (You + 5 friends)
* **Date & Time:** Next Friday at 7:00 PM
* **Cuisine Preference:** Italian

***

### Step 4: Restaurant Booking Call Script Draft

> **Butler:** "Hello, I would like to reserve a table for a party of 6 for next Friday at 7:00 PM under [User's Name]. Do you have availability for outdoor or indoor seating for a group of 6 at that time?"  
> **Host:** "[Restaurant response]"  
> **Butler:** "Wonderful. Please confirm the reservation for 6 people at 7:00 PM. Thank you very much."

***

### Step 5 & 6 Preparation: Confirmation Required

Be

In [15]:
new_thread_id = "butler_session_002"
new_config = {"configurable": {"thread_id": new_thread_id}}

isolated_request = {
    "messages": [
        {
            "role": "user",
            "content": "I need help organizing a birthday party for my colleague Jane. About 20 people."
        }
    ]

}

print("=" * 60)
print(f"THREAD ISOLATION TEST: New thread '{new_thread_id}'")
print("=" * 60)
print("User: I need help organizing a birthday party for my colleague Jane. About 20 people.")
print()
result = agent.invoke(isolated_request, new_config)
response = format_conversation(result)
print("Agent:")
print(response)
print()
print(f"New thread '{new_thread_id}' has NO knowledge of the college friends dinner.")
print("  Each thread is completely isolated.")


THREAD ISOLATION TEST: New thread 'butler_session_002'
User: I need help organizing a birthday party for my colleague Jane. About 20 people.

Agent:
I have initiated the process for Jane's birthday celebration. Here is our progress:

### **Step 1 & 2: Initial Setup & Availability Request Email Drafted**

I have created a draft email to gather availability from the guests:

* **To:** `colleagues@company.com` (Placeholder)
* **Subject:** Jane's Birthday Celebration - Availability Check
* **Body:**
> Hello everyone,
> 
> We are planning a surprise birthday celebration for our colleague Jane for a party of around 20 guests! 
> 
> To help us select the best date and venue, could you please reply with your availability over the upcoming week/weekend, as well as any dietary preferences?
> 
> Please let us know by [Deadline Date].
> 
> Best regards,  
> [Your Name]

---

### **Next Steps:**
To customize and finalize this email before sending, please let me know:
1. **Target dates or weekend op